<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;">
    <img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;">
  </div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">
    MSc. CORO DASSIP
  </div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Image Processing Fundamental — Implementation</b></h1>
</div>

## Setup — Environment and Configuration


In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Noise should change the image, not the experiment itself — fix the RNG so every rerun sees the same corruption.
RNG = np.random.default_rng(42)

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titlesize"] = 11

## 1. Data and Output Paths


In [ ]:
# Find the lab once so every later data/output path works from either common notebook launch location.
def find_lab_root(start: Path) -> Path:
    """Find the lab folder no matter where the notebook was launched.
    
    Sometimes Jupyter starts in the lab folder, sometimes inside notebooks/. Rather
    than hard-code one path, we walk upward until both data/ and notebooks/ are
    present.
    
    From that point on, every file path can be repository-relative and reproducible."""
    start = start.resolve()

    for candidate in [start, *start.parents]:
        # Accept only a parent that satisfies the full lab-directory contract.
        if (candidate / "data").is_dir() and (candidate / "notebooks").is_dir():
            return candidate

    raise FileNotFoundError(
        "Could not locate the lab root. "
        "Expected a directory containing both 'data/' and 'notebooks/'."
    )


# Resolve paths from the actual lab root so the notebook works from multiple launch locations.
LAB_DIR = find_lab_root(Path.cwd())
# Keep source images separate from generated artifacts.
DATA_DIR = LAB_DIR / "data"
# Centralize figures so validation can verify the complete output contract.
OUTPUT_DIR = LAB_DIR / "outputs" / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Map semantic names to source files once so later experiments reuse consistent references.
IMAGE_FILES = {
    "einstein": DATA_DIR / "einstein.png",
    "peppers": DATA_DIR / "peppers.png",
    "ballons": DATA_DIR / "ballons.jpg",
    "grass": DATA_DIR / "grass.jpg",
    "tower": DATA_DIR / "Elizabeth_Tower_London.jpg",
}

missing_files = [path.name for path in IMAGE_FILES.values() if not path.exists()]
assert not missing_files, f"Missing input files: {missing_files}"

print("Lab directory :", LAB_DIR)
print("Data directory:", DATA_DIR)
print("Output folder :", OUTPUT_DIR)
print("Images found  :", len(IMAGE_FILES))

## 2. Sampling and Quantization Experiments


### Sampling Experiment


In [ ]:
# Start with a smooth signal whose structure is known; then any degradation comes from sampling, not unknown image content.
x = np.linspace(0, 2 * np.pi, 256)
y = np.linspace(0, 2 * np.pi, 256)
xx, yy = np.meshgrid(x, y)

continuous_like = (
    0.55
    + 0.25 * np.sin(2.0 * xx)
    + 0.20 * np.cos(3.0 * yy)
)
continuous_like = np.clip(continuous_like, 0.0, 1.0)

# Increase the sampling step aggressively so the transition from detail to aliasing is easy to see.
sampling_steps = [1, 4, 8, 16]

fig, axes = plt.subplots(1, 4, figsize=(14, 3.4))

for ax, step in zip(axes, sampling_steps):
    sampled = continuous_like[::step, ::step]
    ax.imshow(sampled, cmap="gray", vmin=0, vmax=1, interpolation="nearest")
    ax.set_title(f"Sampling step = {step}\nshape = {sampled.shape}")
    ax.axis("off")

fig.suptitle("Spatial Sampling")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "01_sampling.png", dpi=300, bbox_inches="tight")
plt.show()

### Quantization Experiment


In [ ]:
# Use a smooth synthetic ramp because sampling artifacts are easier to isolate without natural-image texture.
gradient = np.tile(np.linspace(0, 1, 512), (90, 1))

# Span binary to full 8-bit quantization so banding disappears progressively.
bit_depths = [1, 2, 4, 8]

fig, axes = plt.subplots(4, 1, figsize=(11, 6))

for ax, bits in zip(axes, bit_depths):
    levels = 2 ** bits

    # Quantization should change intensity precision only, so map the same normalized signal onto fewer allowed levels.
    quantized = np.round(gradient * (levels - 1)) / (levels - 1)

    ax.imshow(quantized, cmap="gray", vmin=0, vmax=1, aspect="auto")
    ax.set_title(f"{bits}-bit quantization → {levels} intensity levels")
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "02_quantization.png", dpi=300, bbox_inches="tight")
plt.show()


## 3. Pixel Coordinates and Array Representation


In [ ]:
# A tiny toy image lets us reason about exact values instead of hiding behavior inside a large photograph.
toy_gray = np.array(
    [
        [0, 32, 64, 96, 128],
        [24, 56, 88, 120, 152],
        [48, 80, 112, 144, 176],
        [72, 104, 136, 168, 208],
        [96, 128, 160, 208, 255],
    ],
    dtype=np.uint8,
)

fig, ax = plt.subplots(figsize=(5, 4.5))
ax.imshow(toy_gray, cmap="gray", vmin=0, vmax=255)

for row in range(toy_gray.shape[0]):
    for col in range(toy_gray.shape[1]):
        ax.text(
            col,
            row,
            str(toy_gray[row, col]),
            ha="center",
            va="center",
            fontsize=8,
        )

ax.set_title("A grayscale image is a matrix of intensities")
ax.set_xlabel("x / column")
ax.set_ylabel("y / row")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "03_grayscale_matrix.png", dpi=300, bbox_inches="tight")
plt.show()

print("Shape:", toy_gray.shape)
print("dtype:", toy_gray.dtype)
print("Pixel at row=2, column=3:", toy_gray[2, 3])

## 4. Image Representation Modes


In [ ]:
# Contrast binary, grayscale and RGB storage with simple controlled examples.
binary = np.zeros((120, 160), dtype=np.uint8)
binary[30:90, 45:120] = 255

# Build a controlled grayscale ramp so bit-depth behavior can be inspected independently of scene content.
grayscale = np.tile(
    np.linspace(0, 255, 160, dtype=np.uint8),
    (120, 1),
)

rgb = np.zeros((120, 160, 3), dtype=np.uint8)
rgb[:, :53] = [255, 0, 0]
rgb[:, 53:106] = [0, 255, 0]
rgb[:, 106:] = [0, 0, 255]

fig, axes = plt.subplots(1, 3, figsize=(11, 3.3))

axes[0].imshow(binary, cmap="gray", vmin=0, vmax=255)
axes[0].set_title(f"Binary\nshape={binary.shape}")

axes[1].imshow(grayscale, cmap="gray", vmin=0, vmax=255)
axes[1].set_title(f"Grayscale\nshape={grayscale.shape}")

axes[2].imshow(rgb)
axes[2].set_title(f"RGB\nshape={rgb.shape}")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "04_image_types.png", dpi=300, bbox_inches="tight")
plt.show()

## 5. Load and Inspect Real Images


In [ ]:
# Convert all reference images the same way so later differences come from processing, not inconsistent loading.
images = {}

for name, path in IMAGE_FILES.items():
    images[name] = np.asarray(Image.open(path).convert("RGB"))

for name, image in images.items():
    print(
        f"{name:9s} | "
        f"shape={str(image.shape):16s} "
        f"dtype={image.dtype} "
        f"range=[{image.min()}, {image.max()}]"
    )

In [ ]:
# See the source set first; understanding the inputs makes every later transformation easier to interpret.
fig, axes = plt.subplots(1, len(images), figsize=(16, 4))

for ax, (name, image) in zip(axes, images.items()):
    ax.imshow(image)
    ax.set_title(f"{name}\n{image.shape[1]}×{image.shape[0]}")
    ax.axis("off")

fig.suptitle("Reference Images")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "05_reference_images.png", dpi=300, bbox_inches="tight")
plt.show()

## 6. Dimensions, Resolution, Aspect Ratio, and Channels


In [ ]:
# Read geometry directly from the array — image size and channel count are data properties, not assumptions.
peppers = images["peppers"]

height, width, channels = peppers.shape
# Pixel count separates spatial sample count from channel count and byte storage.
pixel_count = height * width
# Aspect ratio records image geometry independently of absolute resolution.
aspect_ratio = width / height

print(f"Height       : {height} pixels")
print(f"Width        : {width} pixels")
print(f"Channels     : {channels}")
print(f"Pixel count  : {pixel_count:,}")
print(f"Aspect ratio : {aspect_ratio:.3f}")


## 7. Data Types, Bit Depth, Dynamic Range, and Memory


In [ ]:
# Connect dtype and item size to legal range and in-memory storage cost.
print("dtype:", peppers.dtype)
print("bytes per value:", peppers.dtype.itemsize)
print("array memory:", f"{peppers.nbytes:,} bytes")
print("array memory:", f"{peppers.nbytes / 1024**2:.3f} MiB")

uint8_info = np.iinfo(np.uint8)
print("uint8 range:", uint8_info.min, "to", uint8_info.max)

## 8. Display Scaling and Visualization Control


In [ ]:
# Demonstrate why explicit display limits matter for low-contrast images.
# Restrict values to a narrow mid-gray interval to isolate display-scaling effects.
low_contrast = np.linspace(90, 165, 256, dtype=np.uint8)
low_contrast = np.tile(low_contrast, (120, 1))

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))

axes[0].imshow(low_contrast, cmap="gray")
axes[0].set_title("Automatic display scaling")

axes[1].imshow(low_contrast, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Fixed display range: 0–255")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "06_display_scaling.png", dpi=300, bbox_inches="tight")
plt.show()

## 9. Pixel Access and Safe Modification


In [ ]:
# Edit one known pixel neighborhood without mutating the original image.
ballons = images["ballons"]

y, x = 140, 220
original_pixel = ballons[y, x].copy()

edited_ballons = ballons.copy()

# A six-pixel neighborhood is large enough to make the local pixel edit visible without dominating the image.
radius = 6
edited_ballons[
    y - radius : y + radius + 1,
    x - radius : x + radius + 1,
] = [255, 0, 255]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].imshow(ballons)
axes[0].scatter([x], [y], s=70, facecolors="none", edgecolors="yellow")
axes[0].set_title("Original + selected pixel")

axes[1].imshow(edited_ballons)
axes[1].set_title("Edited copy")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "07_pixel_edit.png", dpi=300, bbox_inches="tight")
plt.show()

print("Selected coordinate (x, y):", (x, y))
print("Stored RGB value:", original_pixel)

## 10. Regions of Interest (ROI)


In [ ]:
# Crop one named rectangle so array indexing becomes a visible spatial operation rather than an abstract slice.
tower = images["tower"]

y0, y1 = 120, 360
x0, x1 = 170, 390

roi = tower[y0:y1, x0:x1]

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

axes[0].imshow(tower)
axes[0].add_patch(
    plt.Rectangle(
        (x0, y0),
        x1 - x0,
        y1 - y0,
        fill=False,
        edgecolor="red",
        linewidth=2,
    )
)
axes[0].set_title("Full image and ROI")

axes[1].imshow(roi)
axes[1].set_title(f"ROI: {roi.shape[1]}×{roi.shape[0]}")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "08_region_of_interest.png", dpi=300, bbox_inches="tight")
plt.show()

## 11. Pixel Neighborhoods


In [ ]:
# Grayscale should follow perceived brightness, so weight green more than blue instead of averaging channels blindly.
einstein_rgb = images["einstein"]

einstein_gray_float = (
    0.299 * einstein_rgb[..., 0].astype(np.float32)
    + 0.587 * einstein_rgb[..., 1].astype(np.float32)
    + 0.114 * einstein_rgb[..., 2].astype(np.float32)
)
einstein_gray = np.clip(einstein_gray_float, 0, 255).astype(np.uint8)

y, x = 120, 120
patch_3x3 = einstein_gray[y - 1 : y + 2, x - 1 : x + 2]

print("Center pixel:", einstein_gray[y, x])
print("3×3 neighborhood:")
print(patch_3x3)

## 12. RGB Channel Decomposition


In [ ]:
# Decompose RGB explicitly to expose channel-specific structure.
red = peppers[..., 0]
green = peppers[..., 1]
blue = peppers[..., 2]

fig, axes = plt.subplots(1, 4, figsize=(14, 4))

axes[0].imshow(peppers)
axes[0].set_title("RGB")

axes[1].imshow(red, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Red channel")

axes[2].imshow(green, cmap="gray", vmin=0, vmax=255)
axes[2].set_title("Green channel")

axes[3].imshow(blue, cmap="gray", vmin=0, vmax=255)
axes[3].set_title("Blue channel")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "09_rgb_channels.png", dpi=300, bbox_inches="tight")
plt.show()

print(
    "Channel means:",
    {
        "R": round(float(red.mean()), 2),
        "G": round(float(green.mean()), 2),
        "B": round(float(blue.mean()), 2),
    },
)


## 13. RGB and BGR Conventions


In [ ]:
# Reverse channel order to illustrate RGB/BGR convention errors.
rgb_example = peppers
bgr_like = rgb_example[..., ::-1]

fig, axes = plt.subplots(1, 2, figsize=(9, 4))

axes[0].imshow(rgb_example)
axes[0].set_title("Correct RGB")

axes[1].imshow(bgr_like)
axes[1].set_title("Channels reversed")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "10_rgb_bgr.png", dpi=300, bbox_inches="tight")
plt.show()

## 14. RGB-to-Grayscale Conversion


In [ ]:
def rgb_to_grayscale(rgb_image: np.ndarray) -> np.ndarray:
    """Collapse RGB into one brightness channel without treating colors equally.
    
    Human vision is more sensitive to green than blue, so a simple channel average
    is not the best luminance estimate. The standard 0.299/0.587/0.114 weighting
    keeps perceived brightness much better.
    
    We promote to floating point first so weighted arithmetic cannot overflow or
    truncate inside uint8."""

    # Promote before arithmetic to avoid unsigned overflow.
    rgb_float = rgb_image.astype(np.float32)

    gray = (
        0.299 * rgb_float[..., 0]
        + 0.587 * rgb_float[..., 1]
        + 0.114 * rgb_float[..., 2]
    )

    return np.clip(gray, 0, 255).astype(np.uint8)


einstein_rgb = images["einstein"]
einstein_gray = rgb_to_grayscale(einstein_rgb)

fig, axes = plt.subplots(1, 2, figsize=(9, 4))

axes[0].imshow(einstein_rgb)
axes[0].set_title(f"RGB shape: {einstein_rgb.shape}")

axes[1].imshow(einstein_gray, cmap="gray", vmin=0, vmax=255)
axes[1].set_title(f"Grayscale shape: {einstein_gray.shape}")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "11_rgb_to_grayscale.png", dpi=300, bbox_inches="tight")
plt.show()


## 15. Image Statistics


In [ ]:
# Summarize intensity distribution with robust descriptive statistics.
grass_gray = rgb_to_grayscale(images["grass"])

statistics = {
    "min": int(grass_gray.min()),
    "max": int(grass_gray.max()),
    "mean": float(grass_gray.mean()),
    "median": float(np.median(grass_gray)),
    "std": float(grass_gray.std()),
    "p05": float(np.percentile(grass_gray, 5)),
    "p95": float(np.percentile(grass_gray, 95)),
}

for name, value in statistics.items():
    print(f"{name:>6s}: {value:.3f}" if isinstance(value, float) else f"{name:>6s}: {value}")

## 16. Intensity Histograms


In [ ]:
# Pair the image with its histogram to connect appearance and intensity counts.
ballons_gray = rgb_to_grayscale(images["ballons"])

counts, bin_edges = np.histogram(
    ballons_gray.ravel(),
    bins=256,
    range=(0, 256),
)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].imshow(ballons_gray, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Grayscale image")
axes[0].axis("off")

axes[1].plot(np.arange(256), counts)
axes[1].set_title("Intensity histogram")
axes[1].set_xlabel("Intensity")
axes[1].set_ylabel("Pixel count")
axes[1].set_xlim(0, 255)

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "12_intensity_histogram.png", dpi=300, bbox_inches="tight")
plt.show()

print("Histogram count:", counts.sum())
print("Number of image pixels:", ballons_gray.size)

In [ ]:
# Shuffle pixels without changing their values: the histogram stays identical even though the image structure disappears.
shuffled = ballons_gray.ravel().copy()
RNG.shuffle(shuffled)
shuffled = shuffled.reshape(ballons_gray.shape)

hist_original, _ = np.histogram(ballons_gray.ravel(), bins=256, range=(0, 256))
hist_shuffled, _ = np.histogram(shuffled.ravel(), bins=256, range=(0, 256))

print("Histograms identical:", np.array_equal(hist_original, hist_shuffled))

fig, axes = plt.subplots(1, 2, figsize=(9, 4))

axes[0].imshow(ballons_gray, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Original")

axes[1].imshow(shuffled, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Pixels shuffled")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()


## 17. Dynamic Range and Min-Max Normalization


In [ ]:
def minmax_normalize(gray_image: np.ndarray) -> np.ndarray:
    """Stretch the intensities that are actually present to the full display range.
    
    If an image only uses a narrow band of gray values, it can look flat even when
    the structure is there. We map the current minimum to 0 and maximum to 255.
    
    A constant image has no range to stretch, so that case is handled explicitly
    instead of dividing by zero."""

    # Promote to float before normalization so subtraction/division cannot wrap in uint8.
image_float = gray_image.astype(np.float32)

    minimum = image_float.min()
    maximum = image_float.max()

    # Constant images have zero dynamic range; return a stable result.
    # Zero dynamic range cannot be stretched; return a stable all-zero image.
    if maximum == minimum:
        return np.zeros_like(gray_image)

    normalized = (image_float - minimum) / (maximum - minimum)
    normalized *= 255.0

    return np.clip(normalized, 0, 255).astype(np.uint8)


source_gray = rgb_to_grayscale(images["ballons"])

# Compress the source into ~30% of the 8-bit range before testing normalization.
low_contrast = 90 + (source_gray.astype(np.float32) / 255.0) * 76
low_contrast = np.clip(low_contrast, 0, 255).astype(np.uint8)

normalized = minmax_normalize(low_contrast)

fig, axes = plt.subplots(2, 2, figsize=(10, 7))

axes[0, 0].imshow(low_contrast, cmap="gray", vmin=0, vmax=255)
axes[0, 0].set_title("Low-contrast image")
axes[0, 0].axis("off")

axes[0, 1].hist(low_contrast.ravel(), bins=256, range=(0, 256))
axes[0, 1].set_title("Before normalization")
axes[0, 1].set_xlabel("Intensity")

axes[1, 0].imshow(normalized, cmap="gray", vmin=0, vmax=255)
axes[1, 0].set_title("After min-max normalization")
axes[1, 0].axis("off")

axes[1, 1].hist(normalized.ravel(), bins=256, range=(0, 256))
axes[1, 1].set_title("After normalization")
axes[1, 1].set_xlabel("Intensity")

fig.tight_layout()
fig.savefig(
    OUTPUT_DIR / "13_dynamic_range_normalization.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

print("Before range:", int(low_contrast.min()), "to", int(low_contrast.max()))
print("After range :", int(normalized.min()), "to", int(normalized.max()))


## 18. `uint8` Arithmetic, Overflow, Clipping, and Floating Point


In [ ]:
value = np.array([250], dtype=np.uint8)

# Demonstrate why unsigned arithmetic must not be used for intermediate sums.
unsafe = value + np.array([20], dtype=np.uint8)

# Do the math where overflow cannot wrap silently; only return to uint8 after the result is safely bounded.
safe_float = value.astype(np.float32) + 20.0
safe_uint8 = np.clip(safe_float, 0, 255).astype(np.uint8)

print("Original value :", value[0])
print("Unsafe result  :", unsafe[0])
print("Safe result    :", safe_uint8[0])

## 19. Noise Model Simulation


In [ ]:
base_gray = einstein_gray

# Corrupt the same clean image in several ways so the noise mechanism — not image content — is what changes.
# Center Gaussian noise at zero so sigma controls spread without also shifting average brightness.
# sigma=20 creates visible additive noise without completely obscuring structure.
gaussian_noise = RNG.normal(0.0, 20.0, size=base_gray.shape)
gaussian = np.clip(
    base_gray.astype(np.float32) + gaussian_noise,
    0,
    255,
).astype(np.uint8)

# Corrupt only a sparse pixel fraction to model impulse noise rather than blur.
salt_pepper = base_gray.copy()
# Corrupt 3% of pixels so impulse noise is sparse but clearly measurable.
probability = 0.03
random_map = RNG.random(base_gray.shape)
salt_pepper[random_map < probability / 2] = 0
salt_pepper[random_map > 1 - probability / 2] = 255

# Normalize to [0,1] before Poisson sampling so signal-dependent noise has interpretable scale.
scaled = base_gray.astype(np.float32) / 255.0
# Scale intensities before Poisson sampling so variance follows signal level.
# Scale=30 gives visible signal-dependent variance while preserving recognizability.
poisson = RNG.poisson(scaled * 30.0) / 30.0
poisson = np.clip(poisson * 255.0, 0, 255).astype(np.uint8)

# Model speckle multiplicatively so stronger signal receives stronger perturbation.
# 0.18 multiplicative spread creates clear speckle without saturating most pixels.
speckle_noise = RNG.normal(0.0, 0.18, size=base_gray.shape)
speckle = base_gray.astype(np.float32) * (1.0 + speckle_noise)
speckle = np.clip(speckle, 0, 255).astype(np.uint8)

fig, axes = plt.subplots(1, 5, figsize=(16, 3.6))

examples = [
    ("Original", base_gray),
    ("Gaussian", gaussian),
    ("Salt & pepper", salt_pepper),
    ("Poisson", poisson),
    ("Speckle", speckle),
]

for ax, (title, image) in zip(axes, examples):
    ax.imshow(image, cmap="gray", vmin=0, vmax=255)
    ax.set_title(title)
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "14_noise_models.png", dpi=300, bbox_inches="tight")
plt.show()


## 20. Image Comparison Metrics


In [ ]:
# Compare the same pixels with several metrics because absolute, squared, and logarithmic errors tell different stories.
def image_metrics(reference: np.ndarray, test: np.ndarray) -> dict:
    """Describe image error from several complementary angles.
    
    MAE asks, "How wrong are pixels on average?" MSE and RMSE punish large mistakes
    more strongly. PSNR expresses the same squared error on a logarithmic scale.
    
    No one number tells the whole story, so we compute them together — and only
    when the two images have exactly the same shape."""

    # Pixel metrics assume one-to-one correspondence; mismatched shapes mean we are no longer comparing the same locations.
    if reference.shape != test.shape:
        raise ValueError("Images must have identical shapes.")

    reference_f = reference.astype(np.float64)
    test_f = test.astype(np.float64)

    difference = reference_f - test_f

    # MAE keeps every pixel error linear, making it less dominated by large outliers than MSE.
mae = np.mean(np.abs(difference))
    # MSE squares deviations so strong local errors contribute disproportionately.
mse = np.mean(difference ** 2)
    # RMSE returns squared-error information to the original intensity unit.
rmse = np.sqrt(mse)

    # Identical images have zero MSE and therefore infinite theoretical PSNR.
    if mse == 0:
        psnr = np.inf
    else:
        psnr = 10.0 * np.log10((255.0 ** 2) / mse)

    return {
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "PSNR": psnr,
    }


# Keep all noisy variants in one named collection so metrics and visual comparisons use the same realizations.
noise_results = {
    "Gaussian": image_metrics(base_gray, gaussian),
    "Salt & pepper": image_metrics(base_gray, salt_pepper),
    "Poisson": image_metrics(base_gray, poisson),
    "Speckle": image_metrics(base_gray, speckle),
}

for noise_name, metrics in noise_results.items():
    print(noise_name)
    for metric_name, value in metrics.items():
        print(f"  {metric_name:>4s}: {value:.4f}")


## 21. Lossless vs Lossy Image Encoding


In [ ]:
example = images["peppers"]

# Encode identical pixels two ways so any reconstruction difference comes from the codec, not the source image.
png_path = OUTPUT_DIR / "15_saved_example.png"
# Save a lossy JPEG beside the lossless PNG so codec-induced changes can be measured directly.
jpg_path = OUTPUT_DIR / "15_saved_example.jpg"

Image.fromarray(example).save(png_path)
# JPEG quality 75 is a moderate-loss setting: artifacts are measurable but not extreme.
Image.fromarray(example).save(jpg_path, quality=75)

png_reload = np.asarray(Image.open(png_path).convert("RGB"))
jpg_reload = np.asarray(Image.open(jpg_path).convert("RGB"))

png_metrics = image_metrics(example, png_reload)
jpg_metrics = image_metrics(example, jpg_reload)

print("PNG reload MSE :", png_metrics["MSE"])
print("JPEG reload MSE:", jpg_metrics["MSE"])
print("PNG path :", png_path)
print("JPEG path:", jpg_path)


## 22. Standard Image Inspection Workflow


In [ ]:
# Provide one reusable inspection routine for unfamiliar image arrays.
def inspect_image(name: str, image: np.ndarray) -> None:
    """Get the facts about an image before doing anything ambitious with it.
    
    Shape tells us the geometry and channels. dtype tells us what arithmetic is
    safe. Min/max and mean reveal the occupied intensity range. Memory size tells
    us the storage cost.
    
    This small inspection step prevents many downstream mistakes that otherwise
    look like algorithm problems."""

    print(f"Name       : {name}")
    print(f"Shape      : {image.shape}")
    print(f"Dimensions : {image.ndim}")
    print(f"dtype      : {image.dtype}")
    print(f"Min / max  : {image.min()} / {image.max()}")
    print(f"Mean       : {image.mean():.3f}")
    print(f"Memory     : {image.nbytes:,} bytes")


inspect_image("peppers", peppers)

## 23. Validation Checks


In [ ]:
# Finish by checking the basic truths every later image-processing notebook will rely on.
assert peppers.ndim == 3
assert peppers.shape[2] == 3
assert einstein_gray.ndim == 2

assert peppers.dtype == np.uint8
assert einstein_gray.dtype == np.uint8

assert counts.sum() == ballons_gray.size

assert normalized.min() == 0
assert normalized.max() == 255

self_metrics = image_metrics(einstein_gray, einstein_gray)
assert self_metrics["MAE"] == 0
assert self_metrics["MSE"] == 0
assert self_metrics["RMSE"] == 0
assert np.isinf(self_metrics["PSNR"])

assert roi.shape[0] == (y1 - y0)
assert roi.shape[1] == (x1 - x0)

print("All fundamental validation checks passed.")


## Final Analysis & Interpretation

### Main findings

- Spatial sampling and intensity quantization remove different kinds of information.
- Shape, channels, dtype, dynamic range, and aspect ratio determine how image arrays can be processed safely.
- RGB decomposition and grayscale conversion show what chromatic information is preserved or discarded.
- Histograms summarize intensity occupancy but do not encode spatial arrangement.
- Different noise models have different statistics and therefore require different restoration strategies.
- MAE, MSE, RMSE, PSNR, and codec comparisons provide complementary views of numerical image degradation.

### Engineering interpretation

An image-processing pipeline begins with representation, not with filtering. Understanding geometry, sampling, channels, dtype, and range prevents many later numerical and conceptual errors.

### Limitations

The noise models are controlled simulations, and full-reference metrics do not fully capture perceptual quality.

### Final conclusion

This notebook establishes the numerical foundations required by the subsequent transformation, filtering, and segmentation labs.
